# 02. ROSE and LROM Cross-Section Comparison

This notebook is the package-driven scaffold for comparing ROSE and LROM at the
cross-section level for $^{40}$Ca$(n,n)$ at $E_{lab}=14.1$ MeV while varying the
optical-potential parameters used in the ROSE paper.

The notebook is intentionally structured around the public `lrom` package. The
current package can already demonstrate the stateful sampling and wavefunction
workflow introduced in Notebook 1; the cross-section observable, CAT metric, and
multi-channel comparison cells below are the contract that the next package update
should fill. Until those APIs exist, future package calls are shown as non-executed
call sketches rather than notebook-local physics code.

Package boundary rule for this notebook: the package owns sampling, model
construction, cross-section prediction, timing, CAT metrics, and artifact/export
state. The notebook owns configuration, explanation, and plotting.

## 1. Imports And Shared Configuration

Keep the notebook imports narrow. `lrom` is the only scientific package interface
used here; no archived `lrom_demo` or notebook-local cross-section implementation is
imported into the active workflow.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "lrom").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for name in list(sys.modules):
    if name == "lrom" or name.startswith("lrom."):
        del sys.modules[name]

import lrom

plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

OUT = ROOT / "outputs" / "notebook02_cross_section_scaffold"
OUT.mkdir(parents=True, exist_ok=True)

print("repo root:", ROOT)
print("lrom version:", getattr(lrom, "__version__", "unknown"))
print("output folder:", OUT)

The physical system is fixed across all comparisons. Multiple partial waves are
required because cross sections are observables assembled from channel-level
scattering information, not from a single radial wavefunction.

In [ ]:
physical_case = {
    "target": (40, 20),
    "projectile": (1, 0),
    "lab_energy": 14.1,
    "channels": tuple(range(0, 11)),
    "observable": "differential_cross_section",
    "angle_degrees": np.linspace(1.0, 179.0, 120),
}

pd.Series(
    {
        "target": "40Ca",
        "projectile": "neutron",
        "E_lab [MeV]": physical_case["lab_energy"],
        "l channels": f"{physical_case['channels'][0]} ... {physical_case['channels'][-1]}",
        "angles": len(physical_case["angle_degrees"]),
    },
    name="Notebook 2 physical case",
)

## 2. Full Optical-Potential Parameter Space

Notebook 2 varies all optical-potential parameters used by the ROSE-paper
cross-section comparison. The exact potential convention should be owned by the
package so that ROSE and LROM see the same named parameters, central values, units,
and training/testing rows.

In [ ]:
optical_parameter_table = pd.DataFrame(
    [
        ("Vv", 46.7238, "MeV", "real volume depth"),
        ("Wv", 1.72334, "MeV", "imaginary volume depth"),
        ("Wd", -7.2357, "MeV", "imaginary surface depth"),
        ("Vso", 6.1, "MeV", "real spin-orbit strength"),
        ("Rv", 4.0538, "fm", "real volume radius"),
        ("Rd", 4.4055, "fm", "surface radius"),
        ("Rso", 1.01 * 40 ** (1.0 / 3.0), "fm", "spin-orbit radius"),
        ("av", 0.6718, "fm", "real volume diffuseness"),
        ("ad", 0.5379, "fm", "surface diffuseness"),
        ("aso", 0.60, "fm", "spin-orbit diffuseness"),
    ],
    columns=["parameter", "central", "unit", "role"],
)
central_parameters = dict(zip(optical_parameter_table["parameter"], optical_parameter_table["central"]))
optical_parameter_table

In [ ]:
training_fraction = 0.20
testing_fraction = 0.20

training_ranges = {
    name: ((1.0 - training_fraction) * value, (1.0 + training_fraction) * value)
    for name, value in central_parameters.items()
}
testing_ranges = {
    name: ((1.0 - testing_fraction) * value, (1.0 + testing_fraction) * value)
    for name, value in central_parameters.items()
}

sampling_contract = pd.DataFrame(
    {
        "central": central_parameters,
        "train low": {key: value[0] for key, value in training_ranges.items()},
        "train high": {key: value[1] for key, value in training_ranges.items()},
        "test low": {key: value[0] for key, value in testing_ranges.items()},
        "test high": {key: value[1] for key, value in testing_ranges.items()},
    }
)
sampling_contract

## 3. Shared Training And Testing Samples

The central scientific constraint is that ROSE and LROM use the same parameter rows.
That means one package-owned sample design is created first, then both model families
consume it. ROSE and LROM must not draw separate Latin-hypercube samples.

In [ ]:
sample_settings = {
    "training_size": 100,
    "testing_size": 80,
    "strategy": "latin_hypercube",
    "seed": 20260524,
    "eim_basis_size": 10,
    "mesh_size": 700,
}

future_sampling_call = """
xs_emulator = lrom.LROM(
    target=(40, 20),
    projectile=(1, 0),
    lab_energy=14.1,
    l=tuple(range(0, 11)),
    fom="nucl-scatter-eq",
    potential="full_woods-saxon",
)
xs_emulator.sampling(
    training_ranges=training_ranges,
    testing_ranges=testing_ranges,
    training_size=100,
    testing_size=80,
    strategy="latin_hypercube",
    seed=20260524,
    mesh_size=700,
    eim_basis_size=10,
)
""".strip()

print(future_sampling_call)

The package should expose enough sample metadata for the notebook to confirm row
identity before training either method.

In [ ]:
future_sample_checks = pd.DataFrame(
    [
        ("training case IDs", "xs_emulator.samples.design.training.case_ids"),
        ("testing case IDs", "xs_emulator.samples.design.testing.case_ids"),
        ("parameter names", "xs_emulator.samples.design.training.parameter_names"),
        ("training values", "xs_emulator.samples.design.training.values"),
        ("testing values", "xs_emulator.samples.design.testing.values"),
    ],
    columns=["needed state", "future package location"],
)
future_sample_checks

## 4. Build Comparable ROSE And LROM Models

The comparison should pair methods by wavefunction basis size. ROSE additionally
varies operator/EIM size `n_U`; predictor LROM varies the number of selected
potential predictors `K`.

In [ ]:
model_grid = pd.DataFrame(
    [
        ("ROSE ROM", 4, 4, None, "same n, small operator basis"),
        ("ROSE ROM", 4, 8, None, "same n, larger operator basis"),
        ("ROSE ROM", 7, 7, None, "matched n and n_U"),
        ("ROSE ROM", 10, 10, None, "larger matched ROSE model"),
        ("ROSE ROM", 15, 15, None, "large ROSE reference point"),
        ("predictor LROM", 4, None, 3, "small predictor set"),
        ("predictor LROM", 4, None, 10, "same n as primary ROSE comparison"),
        ("predictor LROM", 6, None, 10, "larger LROM basis"),
    ],
    columns=["method", "n", "n_U", "K", "purpose"],
)
model_grid

In [ ]:
future_training_call = """
rose_models = xs_emulator.train_rose_cross_section_models(
    basis_operator_pairs=[(4, 4), (4, 8), (7, 7), (10, 10), (15, 15)],
    observable="differential_cross_section",
)
lrom_models = xs_emulator.train_lrom_cross_section_models(
    basis_predictor_pairs=[(4, 3), (4, 10), (6, 10)],
    predictor="potential",
    observable="differential_cross_section",
)
""".strip()

print(future_training_call)

## 5. Cross-Section Prediction Contract

The observable API should return full-order, LS-floor, ROSE ROM, and predictor LROM
cross sections on the same angle grid. The older linear-LROM baseline is deliberately
omitted from Notebook 2.

In [ ]:
future_prediction_call = """
xs_results = xs_emulator.evaluate_cross_sections(
    angles_degrees=physical_case["angle_degrees"],
    include_methods=("fom", "ls", "rose", "lrom"),
    splits=("training", "testing"),
)
""".strip()

expected_xs_result_state = pd.DataFrame(
    [
        ("angles_degrees", "shape (n_angles,)", "common theta_cm grid"),
        ("training.fom", "shape (n_train, n_angles)", "full-order reference"),
        ("training.ls", "shape (n_train, n_angles)", "basis projection floor"),
        ("training.rose[(n, n_U)]", "shape (n_train, n_angles)", "ROSE ROM predictions"),
        ("training.lrom[(n, K)]", "shape (n_train, n_angles)", "predictor LROM predictions"),
        ("testing.fom", "shape (n_test, n_angles)", "held-out full-order reference"),
        ("testing.ls", "shape (n_test, n_angles)", "held-out LS floor"),
        ("testing.rose[(n, n_U)]", "shape (n_test, n_angles)", "held-out ROSE ROM"),
        ("testing.lrom[(n, K)]", "shape (n_test, n_angles)", "held-out predictor LROM"),
    ],
    columns=["state", "expected shape", "meaning"],
)

print(future_prediction_call)
expected_xs_result_state

## 6. Representative Cross-Section Curves

This figure should show where the CAT-plot and violin summaries come from:
observable curves over angle for a few held-out parameter rows. The primary visual
comparison is FOM, LS floor, ROSE ROM, and predictor LROM.

In [ ]:
representative_test_indices = [51, 63, 79]

future_representative_plot = """
representative = xs_results.testing.select_cases(indices=representative_test_indices)

for case in representative:
    ax.plot(xs_results.angles_degrees, case.fom, color="black", label="FOM")
    ax.plot(xs_results.angles_degrees, case.ls[4], "--", color="0.45", label="LS floor (n=4)")
    ax.plot(xs_results.angles_degrees, case.rose[(4, 10)], "--", color="tab:blue", label="ROSE ROM (n=4, n_U=10)")
    ax.plot(xs_results.angles_degrees, case.lrom[(4, 10)], "-", color="tab:orange", label="predictor LROM (n=4, K=10)")
    ax.set_yscale("log")
    ax.set_xlabel(r"$\\theta_{cm}$ [deg]")
    ax.set_ylabel(r"$d\\sigma/d\\Omega$ [mb/sr]")
""".strip()

print(future_representative_plot)
pd.DataFrame({"representative testing row": representative_test_indices})

## 7. Cross-Section Error Distributions

The error distributions compare methods over the same all-parameter training and
testing rows. The default metric is median pointwise relative cross-section error
over angle; relative L2 cross-section error can be shown beside it when available.

In [ ]:
future_error_distribution_call = """
xs_errors = xs_emulator.cross_section_errors(
    xs_results,
    metrics=("median_pointwise_relative", "relative_l2"),
    methods=(
        ("ls", {"n": 4}),
        ("rose", {"n": 4, "n_U": 10}),
        ("lrom", {"n": 4, "K": 10}),
    ),
)
# Use xs_errors arrays below with direct matplotlib violin/scatter code in this notebook.
""".strip()

expected_error_table = pd.DataFrame(
    [
        ("LS floor", "n=4", "basis projection cross-section floor"),
        ("ROSE ROM", "n=4, n_U=10", "standard ROSE cross-section ROM"),
        ("predictor LROM", "n=4, K=10", "potential-predictor LROM"),
    ],
    columns=["curve", "configuration", "role"],
)

print(future_error_distribution_call)
expected_error_table

## 8. CAT Plot: Accuracy Versus Online Time

The CAT plot places each held-out parameter row in accuracy-time space. The y-axis is
median pointwise relative cross-section error. The x-axis is online time per sample:
parameter values in, cross section out. Predictor evaluation is included for LROM;
precomputable training/offline quantities are not.

In [ ]:
future_cat_call = """
cat_results = xs_emulator.cross_section_cat(
    xs_results,
    timing="online_per_sample",
    accuracy="median_pointwise_relative",
    million_samples_per_hour_line=True,
    error_reference_lines=(0.10,),
)

# Notebook-owned plotting example:
# fig, ax = plt.subplots(figsize=(8, 5))
# for label, cloud in cat_results.point_clouds.items():
#     ax.scatter(cloud.times, cloud.errors, label=label)
# ax.axhline(0.10, color="black", linestyle="--")
# ax.axvline(3600 / 1_000_000, color="black", linestyle="--")
# ax.set_xscale("log")
# ax.set_yscale("log")
""".strip()

cat_plot_contract = pd.DataFrame(
    [
        ("x", "online time per sample [s]"),
        ("y", "median pointwise relative cross-section error"),
        ("horizontal reference", "10 percent error"),
        ("vertical reference", "1 million samples/hour"),
        ("point", "one method/configuration on one test sample"),
    ],
    columns=["plot element", "meaning"],
)

print(future_cat_call)
cat_plot_contract


## 9. Rainbow Plots And Predictor Locations

This optional diagnostic connects the cross-section comparison back to the physical
parameter space. It should show potential rainbows and where potential-predictor LROM
places selected radii or operator-informed sample locations for representative
channels.

In [ ]:
future_rainbow_call = """
diagnostics = xs_emulator.cross_section_diagnostics(
    channels=(0, 3, 6, 10),
    parameter_rows="testing_representatives",
    include=("potential_rainbow", "predictor_locations"),
)

# Notebook-owned plotting example:
# fig, ax = plt.subplots(figsize=(7, 4))
# for row in diagnostics.potential_rainbows:
#     ax.plot(row.radius, row.potential.real, alpha=0.2)
# ax.scatter(diagnostics.predictor_locations.radius, diagnostics.predictor_locations.value)
""".strip()

print(future_rainbow_call)


## 10. Package API Checklist

Filling this notebook should require package additions, not notebook-local physics.

- `potential="full_woods-saxon"` or an equivalent named schema for the full optical-potential parameter set.
- Multi-channel sampling and training for fixed target/projectile/energy cases.
- Shared sample design state that both ROSE and LROM consume without resampling.
- Package-owned cross-section assembly from channel-level model outputs.
- Cross-section prediction state for FOM, LS floor, ROSE ROM, and predictor LROM.
- Cross-section error metrics over training and testing splits.
- CAT timing helpers that time only online prediction.
- Diagnostic arrays for potential rainbows and selected predictor locations; plotting stays in notebook cells.
- Optional export hook for a trained cross-section emulator artifact or interactive HTML view.